# Proyecto final · Análisis de actividad antifúngica

**Versión lista para ejecutar.**

Trabajamos con los ensayos *in vitro* de inhibición del crecimiento micelial y con la
caracterización DLS de las formulaciones, contenidos en
`resultados actividad antifúngica.xlsx`.

| Bloque | Producto |
|---|---|
| 1. Conexión con los datos | El Excel cargado y verificado en pandas |
| 2. Skills | El paquete completo de `scientific-agent-skills` instalado en `.agents/skills/` |
| 3. Limpieza | Tablas *tidy* de actividad y de DLS |
| 4. Descriptiva | Medias, desvíos y auditoría de faltantes por grupo |
| 5. Bioestadística | Inhibición, supuestos, tests, dosis-respuesta 4PL y sensibilidad |
| 6. Visualización | Figuras guardadas en `outputs/` |
| 7. IR | Bloque preparado para las formulaciones ≥ 8 (datos aún no disponibles) |


**Advertencias que atraviesan todo el análisis.**

- El control de crecimiento tiene **una sola medición** (1.9 cm): todos los `% de
  inhibición` son condicionales a ese valor y su incertidumbre no es estimable.
- Las 3 celdas `NA` son faltantes declarados; los 5 ceros son **inhibición completa**,
  un dato válido que se evalúa en el análisis de sensibilidad.
- Solo 2 dosis en T5–T8: no permite estimar una curva dosis-respuesta creíble.
- El análisis es **exploratorio**: los valores de *p* se reportan como descriptivos.


## Estado: listo para ejecutar, pendiente de planillas adicionales

Esta versión tiene **todo el código cargado y ninguna celda ejecutada**: las salidas
están vacías y el contador de ejecución está en cero.

Antes de apretar *Run All*:

1. Agregar las planillas que faltan (formulaciones ≥ 8, datos de IR, etc.). Se pueden
   poner como **hojas nuevas del mismo Excel** o como archivos separados en la carpeta.
2. Revisar `RUTA_EXCEL`, `HOJA_ACTIVIDAD` y `HOJA_DLS` en la celda de configuración.
3. Ejecutar el cuaderno de arriba abajo. La celda 1.1 avisa qué hojas adicionales
   encontró y el bloque 7 carga los archivos de IR cuando estén disponibles.


## 0. Configuración

Esta es la única celda que nombra archivos, hojas y constantes. El resto del
cuaderno usa nombres genéricos (`tidy`, `dls_tidy`, `x`, `y`). Para trabajar con
otro archivo, se edita acá y se vuelve a ejecutar todo.


In [ ]:
# === Configuración del conjunto de datos ===
from pathlib import Path
import hashlib
import re
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", context="paper")

RUTA_EXCEL = Path("resultados actividad antifúngica.xlsx")
HOJA_ACTIVIDAD = "Actividad Antifúngica"
HOJA_DLS = "DLS"
HOJAS_ESPERADAS = (HOJA_ACTIVIDAD, HOJA_DLS)
CRECIMIENTO_CONTROL_CM = 1.9     # cm; una sola medición, sin réplicas
SEMILLA = 20261126
N_BOOT = 2000
ALFA = 0.05
PATRON_IR = "ir_*.csv"           # formulaciones ≥ 8: archivos aún no disponibles
CARPETA_SALIDA = Path("outputs")
CARPETA_SALIDA.mkdir(exist_ok=True)

def sha256(ruta):
    """Huella del archivo original, que nunca se modifica."""
    h = hashlib.sha256()
    with open(ruta, "rb") as manejador:
        for bloque in iter(lambda: manejador.read(65536), b""):
            h.update(bloque)
    return h.hexdigest()

rng = np.random.default_rng(SEMILLA)
print("Excel:", RUTA_EXCEL.resolve())
print("SHA-256:", sha256(RUTA_EXCEL))

## 1. Conectar a la IA con la base de datos de antifúngicos

**Producto:** el Excel leído en pandas, con sus dos hojas identificadas y verificadas.

La "conexión" consiste en poner los datos en un DataFrame que el asistente pueda
leer y discutir. No se copia ni se edita el archivo original: se registra su
SHA-256 para saber que siempre trabajamos sobre la misma versión.


### Consigna 1.1 · Leer el archivo

Cargá el Excel con `pd.ExcelFile`, listá las hojas y lee cada una en un DataFrame.
Guardá la hoja de actividad en `actividad_raw` y la de DLS en `dls_raw` (esta última
sin encabezado, porque tiene dos filas de títulos).

**Se espera:** la lista de hojas y las formas `(filas, columnas)` de los dos DataFrames.


In [ ]:
# El archivo original solo se lee; nunca se escribe.
excel = pd.ExcelFile(RUTA_EXCEL)
print("Hojas disponibles:", excel.sheet_names)

actividad_raw = pd.read_excel(excel, sheet_name=HOJA_ACTIVIDAD)
dls_raw = pd.read_excel(excel, sheet_name=HOJA_DLS, header=None)

print("Actividad:", actividad_raw.shape)
print("DLS:", dls_raw.shape)
actividad_raw.head()

# Planillas nuevas: se detectan solas, sin alterar las dos hojas conocidas.
hojas_extra = {
    h: pd.read_excel(excel, sheet_name=h, header=None)
    for h in excel.sheet_names if h not in HOJAS_ESPERADAS
}
if hojas_extra:
    print("Hojas adicionales detectadas:", {h: v.shape for h, v in hojas_extra.items()})
else:
    print("No hay hojas adicionales todavía (se esperan formulaciones ≥ 8).")

### Consigna 1.2 · Verificar antes de seguir

Comprobá con código —no a ojo— que los encabezados de la hoja de actividad son los
esperados y que la huella del archivo coincide con la que se registró en la
configuración.

**Se espera:** un `assert` que pase y el SHA-256 impreso.


In [ ]:
assert list(actividad_raw.columns) == ["TRATAMIENTO", "Crecimiento radial en cm del hongo"]
assert HOJA_ACTIVIDAD in excel.sheet_names and HOJA_DLS in excel.sheet_names

print("Encabezados y hojas verificados.")
print("SHA-256 del original:", sha256(RUTA_EXCEL))
print("Filas de actividad:", len(actividad_raw), "| filas de DLS:", len(dls_raw))

## 2. Descargar un paquete de skills

**Producto:** todas las skills de `K-Dense-AI/scientific-agent-skills` instaladas en
`.agents/skills/`.

El link original del cuaderno (`nazarenacorvalan-sys/Proyecto_final_antifungicos`)
hoy devuelve 404. Se usa en su lugar el repositorio que ya fijó el curso en su
`skills-lock.json`, con el mismo *commit* para que la descarga sea reproducible.


### Consigna 2.1 · Instalar y contar

Descargá el zip del repositorio fijado en el commit `c21d13b…`, extraé **solo** el
directorio `skills/` dentro de `.agents/skills/`, y contá cuántos archivos
`SKILL.md` quedaron. Volver a ejecutar la celda no debe duplicar nada.

**Se espera:** un mensaje con la cantidad de skills instaladas y un `assert` de que no
sea cero.


In [ ]:
URL_SKILLS = ("https://codeload.github.com/K-Dense-AI/scientific-agent-skills/"
               "zip/c21d13b3cc2894680229ba9959a53281acb9200e")
DESTINO_SKILLS = Path(".agents/skills")
DESTINO_SKILLS.mkdir(parents=True, exist_ok=True)

with urllib.request.urlopen(URL_SKILLS, timeout=180) as respuesta:
    contenido_zip = respuesta.read()

with zipfile.ZipFile(io.BytesIO(contenido_zip)) as archivo_zip:
    raices = {e.filename.split("/")[0] for e in archivo_zip.infolist()
              if "/" in e.filename and e.filename.startswith("scientific-agent-skills-")}
    assert len(raices) == 1, f"Raíces inesperadas en el zip: {raices}"
    prefijo = raices.pop() + "/skills/"
    cantidad_skills = 0
    for entrada in archivo_zip.infolist():
        if not entrada.filename.startswith(prefijo):
            continue
        relativa = entrada.filename[len(prefijo):]
        if not relativa:
            continue
        destino = DESTINO_SKILLS / relativa
        if entrada.is_dir():
            destino.mkdir(parents=True, exist_ok=True)
        else:
            destino.parent.mkdir(parents=True, exist_ok=True)
            destino.write_bytes(archivo_zip.read(entrada))
            if destino.name == "SKILL.md":
                cantidad_skills += 1

print(f"Skills instaladas en {DESTINO_SKILLS}: {cantidad_skills}")
assert cantidad_skills > 0, "No se encontraron archivos SKILL.md en el paquete descargado"

## 3. Limpieza: de celdas fusionadas a tablas *tidy*

La hoja de actividad tiene 20 bloques de tres filas con la etiqueta de tratamiento
en la primera (celdas fusionadas). La hoja DLS tiene dos filas de encabezado,
coma decimal, el símbolo `±` y valores `ND`. Cada hoja necesita su own lectura.


### Consigna 3.1 · Actividad antifúngica

 Propagá la etiqueta hacia las filas en blanco, convertí los `NA` en faltantes,
separá `tratamiento` y `dosis` con una expresión regular y conservá los ceros.
La unidad de análisis es la réplica biológica: una fila por medición.

**Se espera:** un DataFrame `tidy` con las columnas `grupo`, `familia`,
`dosis_ug_ml`, `crecimiento_cm`, `replica_id` y `es_control`, con el control
codificado como dosis 0. Imprimir cantidad de filas, faltantes y ceros.


In [ ]:
actividad = actividad_raw.copy()
actividad.columns = ["tratamiento_original", "crecimiento_cm"]
actividad["tratamiento"] = actividad["tratamiento_original"].ffill().str.strip()

# 'NA' es un faltante declarado; los ceros son inhibición completa (dato válido).
actividad["crecimiento_cm"] = pd.to_numeric(
    actividad["crecimiento_cm"].replace("NA", np.nan), errors="coerce"
)

es_control = actividad["tratamiento"].str.upper().eq("CONTROL CRECIMIENTO")
patron = re.compile(r"^(T\d+)\s+(\d+)\s*µg/ml$", flags=re.IGNORECASE)

def extraer(texto):
    encontrado = patron.match(texto)
    return (encontrado.group(1), float(encontrado.group(2))) if encontrado else (None, np.nan)

partes = actividad.loc[~es_control, "tratamiento"].map(extraer)
actividad["familia"] = "CONTROL"
actividad["dosis_ug_ml"] = 0.0
actividad.loc[~es_control, "familia"] = [p[0] for p in partes]
actividad.loc[~es_control, "dosis_ug_ml"] = [p[1] for p in partes]
actividad["es_control"] = es_control
actividad["grupo"] = np.where(
    es_control,
    "CONTROL",
    actividad["familia"] + " " + actividad["dosis_ug_ml"].map(lambda d: f"{d:.0f} µg/ml"),
)
actividad["replica_id"] = actividad.groupby("grupo").cumcount() + 1

tidy = actividad[[
    "grupo", "familia", "dosis_ug_ml", "crecimiento_cm", "replica_id", "es_control"
]].copy()

print("Filas tidy:", len(tidy), "| grupos:", tidy["grupo"].nunique())
print("Faltantes:", int(tidy["crecimiento_cm"].isna().sum()),
      "| ceros:", int((tidy["crecimiento_cm"] == 0).sum()))
tidy.head(8)

### Consigna 3.2 · DLS

Parseá la hoja `DLS`: encontrá la primera fila de datos, separá `"792,5 ± 4,9"`
en media y desvío, pasá la coma decimal a punto y convertí `ND` en faltante.

**Se espera:** `dls_tidy` con una fila por formulación (T5–T8), tamaño y desvío en
`t0` y `t30`, y los dos índices de polidispersidad.


In [ ]:
inicio = dls_raw.index[
    dls_raw.iloc[:, 0].astype(str).str.strip().str.match(r"^T\d+$")
].min()

def partir_tamano(valor):
    texto = str(valor).strip().replace(",", ".")
    if texto.upper() in {"ND", "NAN", ""}:
        return np.nan, np.nan
    media, desvio = texto.split("±")
    return float(media.strip()), float(desvio.strip())

def a_numero(valor):
    return pd.to_numeric(str(valor).strip().replace(",", "."), errors="coerce")

registros = []
for _, fila in dls_raw.loc[inicio:].iterrows():
    tamano_t0, desvio_t0 = partir_tamano(fila.iloc[1])
    tamano_t30, desvio_t30 = partir_tamano(fila.iloc[3])
    registros.append({
        "tratamiento": str(fila.iloc[0]).strip(),
        "tamano_t0_nm": tamano_t0,
        "desvio_t0_nm": desvio_t0,
        "pdi_t0": a_numero(fila.iloc[2]),
        "tamano_t30_nm": tamano_t30,
        "desvio_t30_nm": desvio_t30,
        "pdi_t30": a_numero(fila.iloc[4]),
    })

dls_tidy = pd.DataFrame(registros)
print("Filas de DLS:", len(dls_tidy))
dls_tidy

## 4. Estadística descriptiva

**Producto:** una tabla resumen por grupo y una auditoría de faltantes.

El control se muestra aparte: tiene una sola medición, así que no tiene desvío.


### Consigna 4.1 · Descriptivos y faltantes

Agregá `crecimiento_cm` por grupo con n, media, mediana, desvío, mínimo y máximo,
y agregá el crecimiento relativo al control. Después mostrá cuántas observaciones
y cuántos faltantes hay en cada combinación familia × dosis.

**Se espera:** 20 filas en la tabla de descriptivos. Las proporciones de la auditoría
deben sumar el total de filas de `tidy`.


In [ ]:
grupos = tidy[~tidy["es_control"]]

descriptivos = (
    grupos.groupby("grupo")["crecimiento_cm"]
    .agg(n="count", media="mean", mediana="median", desvio="std", minimo="min", maximo="max")
    .round(3)
)
descriptivos["crecimiento_relativo"] = (descriptivos["media"] / CRECIMIENTO_CONTROL_CM).round(3)

print("Control (una sola medición):", CRECIMIENTO_CONTROL_CM, "cm")
print("Grupos en la tabla:", len(descriptivos))
descriptivos

In [ ]:
auditoria = tidy.groupby(["familia", "dosis_ug_ml"])["crecimiento_cm"].agg(
    n="count", faltantes=lambda s: int(s.isna().sum())
)
print(auditoria.to_string())
assert auditoria["n"].sum() + auditoria["faltantes"].sum() == len(tidy)

## 5. Bioestadística

**Producto:** porcentajes de inhibición, decisión de tests por familia, comparaciones
pareadas, curvas dosis-respuesta y análisis de sensibilidad.

Todo se trata como exploratorio. La decisión del test se toma **antes** de mirar
los resultados, usando normalidad sobre los residuos del modelo y homocedasticidad
con Levene centrado en la mediana.


### Consigna 5.1 · Inhibición y supuestos

Calculá `% inhibición = 100·(1 − g/1.9)` y evaluá los supuestos dentro de cada
familia T1–T8. Usá Shapiro-Wilk sobre los residuos (restar la media de cada dosis)
y Levene con centro = mediana entre las dosis.

**Se espera:** una tabla `supuestos` con n, p de Shapiro, p de Levene y las dos
decisiones binarias con `α = 0.05`.


In [ ]:
tidy["inhibicion_pct"] = 100 * (1 - tidy["crecimiento_cm"] / CRECIMIENTO_CONTROL_CM)

filas_supuestos = []
for familia, sub in grupos.groupby("familia"):
    residuos = sub["crecimiento_cm"] - sub.groupby("dosis_ug_ml")["crecimiento_cm"].transform("mean")
    residuos = residuos.dropna()
    p_norm = stats.shapiro(residuos).pvalue if len(residuos) >= 3 else np.nan
    muestras = [g["crecimiento_cm"].dropna().values for _, g in sub.groupby("dosis_ug_ml")]
    p_homo = stats.levene(*muestras, center="median").pvalue if len(muestras) >= 2 else np.nan
    filas_supuestos.append({
        "familia": familia,
        "n": len(sub),
        "shapiro_p": round(float(p_norm), 4) if np.isfinite(p_norm) else np.nan,
        "levene_p": round(float(p_homo), 4) if np.isfinite(p_homo) else np.nan,
        "normal": bool(p_norm >= ALFA) if np.isfinite(p_norm) else None,
        "homocedastico": bool(p_homo >= ALFA) if np.isfinite(p_homo) else None,
    })

supuestos = pd.DataFrame(filas_supuestos)
decision = {f: ("ANOVA + Tukey" if r["normal"] and r["homocedastico"]
                else "Welch + t de Welch (Holm)" if r["normal"]
                else "Kruskal-Wallis + Mann-Whitney (Holm)")
            for f, r in supuestos.set_index("familia").iterrows()}
supuestos["test_elegido"] = supuestos["familia"].map(decision)
print("Shapiro sobre residuos; Levene con centro = mediana; α =", ALFA)
supuestos

### Consigna 5.2 · Tests por familia

Aplicá la regla anterior: ANOVA + Tukey si hay normalidad y homocedasticidad;
Welch ANOVA + comparaciones de Welch con Holm si solo falla la homocedasticidad;
Kruskal-Wallis + Mann-Whitney con Holm en los demás casos.

**Se espera:** `resultados` con el test ómnibus por familia y `comparaciones` con
cada par de dosis, su *p* cruda, su *p* ajustada y el flag de significancia.


In [ ]:
def welch_anova(*muestras):
    """Welch ANOVA sin dependencias externas."""
    n = np.array([len(g) for g in muestras], dtype=float)
    medias = np.array([np.mean(g) for g in muestras])
    varianzas = np.array([max(np.var(g, ddof=1), 1e-12) for g in muestras])
    w = n / varianzas
    sw = w.sum()
    media_ponderada = (w * medias).sum() / sw
    k = len(muestras)
    termino = np.sum((1 - w / sw) ** 2 / (n - 1))
    f = ((w * (medias - media_ponderada) ** 2).sum() / (k - 1)) / (1 + (2 * (k - 2) / (k**2 - 1)) * termino)
    df1 = k - 1
    df2 = (k**2 - 1) / (3 * termino)
    return f, stats.f.sf(f, df1, df2)

resultados, comparaciones = [], []
supuestos_idx = supuestos.set_index("familia")

for familia, sub in grupos.groupby("familia"):
    niveles = sorted(sub["dosis_ug_ml"].unique())
    muestras = [sub.loc[sub["dosis_ug_ml"] == d, "crecimiento_cm"].dropna().values for d in niveles]
    fila_sup = supuestos_idx.loc[familia]

    if fila_sup["normal"] and fila_sup["homocedastico"]:
        nombre_omnibus, accion = "ANOVA de un factor", lambda a, b: stats.ttest_ind(a, b, equal_var=True)
        estadistico, p = stats.f_oneway(*muestras)
        nombre_par = "t de Student"
    elif fila_sup["normal"]:
        nombre_omnibus, accion = "Welch ANOVA", lambda a, b: stats.ttest_ind(a, b, equal_var=False)
        estadistico, p = welch_anova(*muestras)
        nombre_par = "t de Welch"
    else:
        nombre_omnibus, accion = "Kruskal-Wallis", lambda a, b: stats.mannwhitneyu(a, b, alternative="two-sided")
        estadistico, p = stats.kruskal(*muestras)
        nombre_par = "Mann-Whitney"

    resultados.append({"familia": familia, "test": nombre_omnibus,
                       "estadistico": round(float(estadistico), 4), "p": round(float(p), 6)})

    pares = []
    for i in range(len(niveles)):
        for j in range(i + 1, len(niveles)):
            stat_par, p_par = accion(muestras[i], muestras[j])
            pares.append({"familia": familia, "dosis_a": niveles[i], "dosis_b": niveles[j],
                          "test": nombre_par, "p_crudo": float(p_par)})
    if pares:
        p_ajustados = multipletests([p_["p_crudo"] for p_ in pares], method="holm")[1]
        for par, p_adj in zip(pares, p_ajustados):
            par["p_holm"] = round(float(p_adj), 6)
            par["significativo"] = bool(p_adj < ALFA)
        comparaciones.extend(pares)

resultados = pd.DataFrame(resultados)
comparaciones = pd.DataFrame(comparaciones)
print("Tests ómnibus:")
print(resultados.to_string(index=False))
print("\nComparaciones:")
print(comparaciones.to_string(index=False))

### Consigna 5.3 · Curvas dosis-respuesta (4PL)

Ajustá la logística de cuatro parámetros para T1–T4, con la asíntota superior fija
en el control (1.9 cm). Si el grupo tiene ceros, fijá la inferior en 0; si no,
estimala entre 0 y 1.9. Obtené el IC 95% de la EC50 por bootstrap de réplicas
dentro de cada dosis, con la semilla fija.

**Se espera:** una tabla con EC50, pendiente, R² e IC; `EC50 no estimable` cuando
el ajuste no converge o el IC no es finito. T5–T8 se excluyen por tener solo 2 dosis.


In [ ]:
def curva_4pl(x, bottom, top, ec50, hill):
    return bottom + (top - bottom) / (1 + (ec50 / x) ** hill)

ajustes_4pl, curvas_4pl = [], {}

for familia in ["T1", "T2", "T3", "T4"]:
    sub = grupos[(grupos["familia"] == familia)].dropna(subset=["crecimiento_cm"])
    por_dosis = {float(d): g["crecimiento_cm"].values
                 for d, g in sub.groupby("dosis_ug_ml")}
    x = np.concatenate([np.full(len(v), d) for d, v in por_dosis.items()])
    y = np.concatenate(list(por_dosis.values()))
    tiene_ceros = bool((y == 0).any())

    def ajustar(xd, yd):
        if tiene_ceros:
            def modelo(xv, ec50, hill):
                return curva_4pl(xv, 0.0, CRECIMIENTO_CONTROL_CM, ec50, hill)
            popt, _ = curve_fit(modelo, xd, yd, p0=[30.0, 1.5],
                                bounds=([1e-3, 0.1], [1e4, 20.0]), maxfev=20000)
            return 0.0, float(popt[0]), float(popt[1])
        def modelo(xv, bottom, ec50, hill):
            return curva_4pl(xv, bottom, CRECIMIENTO_CONTROL_CM, ec50, hill)
        popt, _ = curve_fit(modelo, xd, yd, p0=[0.1, 30.0, 1.5],
                            bounds=([0.0, 1e-3, 0.1], [CRECIMIENTO_CONTROL_CM, 1e4, 20.0]),
                            maxfev=20000)
        return float(popt[0]), float(popt[1]), float(popt[2])

    bottom, ec50, hill = ajustar(x, y)
    y_pred = curva_4pl(x, bottom, CRECIMIENTO_CONTROL_CM, ec50, hill)
    r2 = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

    boots = []
    for _ in range(N_BOOT):
        xb = np.concatenate([np.full(len(v), d) for d, v in por_dosis.items()])
        yb = np.concatenate([v[rng.integers(0, len(v), len(v))] for v in por_dosis.values()])
        try:
            boots.append(ajustar(xb, yb)[1])
        except Exception:
            continue

    suficiente = len(boots) >= 0.5 * N_BOOT and bool(np.isfinite(boots).all())
    ic = tuple(np.percentile(boots, [2.5, 97.5])) if suficiente else (np.nan, np.nan)
    ajustes_4pl.append({
        "familia": familia, "bottom_cm": round(bottom, 4), "top_cm": CRECIMIENTO_CONTROL_CM,
        "ec50_ug_ml": round(ec50, 2), "hill": round(hill, 3), "r2": round(float(r2), 4),
        "ic_inf": round(float(ic[0]), 2), "ic_sup": round(float(ic[1]), 2),
        "estimable": suficiente, "boots_ok": len(boots),
    })
    if suficiente:
        curvas_4pl[familia] = (bottom, ec50, hill)

ajustes_4pl = pd.DataFrame(ajustes_4pl)
print("EC50 de T1–T4 (T5–T8 no estimables: solo 2 dosis)")
ajustes_4pl

### Consigna 5.4 · Sensibilidad de los ceros

Repetí el resumen de los grupos con ceros bajo dos escenarios: ceros como dato
válido (análisis principal) y ceros reemplazados por ε = 0.01 cm.

**Se espera:** una tabla corta que muestre que la conclusión cualitativa no cambia.


In [ ]:
sensibilidad = []
sub_ceros = tidy[tidy["grupo"].isin(["T3 80 µg/ml", "T4 80 µg/ml"])].copy()

for escenario, reemplazo in [("ceros válidos", None), ("ceros = 0.01 cm", 0.01)]:
    sub = sub_ceros.copy()
    if reemplazo is not None:
        sub.loc[sub["crecimiento_cm"] == 0, "crecimiento_cm"] = reemplazo
    sub["inhibicion_pct"] = 100 * (1 - sub["crecimiento_cm"] / CRECIMIENTO_CONTROL_CM)
    for grupo, fila in sub.groupby("grupo"):
        sensibilidad.append({
            "escenario": escenario, "grupo": grupo,
            "media_cm": round(fila["crecimiento_cm"].mean(), 3),
            "inhibicion_pct": round(fila["inhibicion_pct"].mean(), 1),
        })

sensibilidad = pd.DataFrame(sensibilidad)
print(sensibilidad.to_string(index=False))
print("\nConclusión: ambos escenarios muestran inhibición completa o casi completa en T3 80 y T4 80.")

## 6. Visualización

**Producto:** figuras guardadas en `outputs/`, con ejes rotulados y unidades.

Se usan puntos individuales sobre las barras para que se vea la réplica biológica,
barras de error de un desvío estándar y letras de significancia para las dosis de
T1–T4.


### Consigna 6.1 · Figura de inhibición

Graficá el `% de inhibición` medio por grupo con barras de error DE, los puntos
individuales encima y letras compactas derivadas de `comparaciones`. Guardá en
PNG a 300 dpi y en PDF.

**Se espera:** un archivo `outputs/fig1_inhibicion.png` y su versión `.pdf`, con la
barra del control en 0%.


In [ ]:
def letras_para_familia(familia, comparaciones, niveles):
    significativas = {
        frozenset((r.dosis_a, r.dosis_b))
        for r in comparaciones.itertuples()
        if r.familia == familia and r.significativo
    }
    asignadas = {nivel: set() for nivel in niveles}
    for nivel in niveles:
        for letra in "abcdefgh":
            if all(frozenset((nivel, otro)) not in significativas
                   for otro in niveles if otro != nivel and letra in asignadas[otro]):
                asignadas[nivel].add(letra)
                break
    return {nivel: "".join(sorted(letras)) for nivel, letras in asignadas.items()}

letras_por_grupo = {}
for familia in ["T1", "T2", "T3", "T4"]:
    sub = tidy[(tidy["familia"] == familia) & ~tidy["es_control"]]
    niveles = sorted(sub["dosis_ug_ml"].unique())
    mapa = letras_para_familia(familia, comparaciones, niveles)
    for dosis, letra in mapa.items():
        grupo = sub.loc[sub["dosis_ug_ml"] == dosis, "grupo"].iloc[0]
        letras_por_grupo[grupo] = letra

resumen = (tidy[~tidy["es_control"]].groupby("grupo")["inhibicion_pct"]
           .agg(media="mean", desvio="std", n="count").reset_index())
resumen["letra"] = resumen["grupo"].map(letras_por_grupo).fillna("")
orden = resumen.sort_values("grupo")["grupo"].tolist()

fig, eje = plt.subplots(figsize=(10, 5))
sns.barplot(data=resumen, x="grupo", y="media", order=orden, color="#4C78A8",
            errorbar=None, ax=eje)
eje.errorbar(x=range(len(orden)), y=resumen.set_index("grupo").loc[orden, "media"],
             yerr=resumen.set_index("grupo").loc[orden, "desvio"].fillna(0),
             fmt="none", ecolor="black", capsize=3, lw=1, ax=eje)
sns.stripplot(data=tidy[~tidy["es_control"]], x="grupo", y="inhibicion_pct",
              order=orden, color="black", size=4, alpha=0.75, jitter=0.12, ax=eje)
eje.axhline(0, color="0.4", lw=1)
for i, grupo in enumerate(orden):
    eje.text(i, resumen.set_index("grupo").loc[grupo, "media"] + 4,
             resumen.set_index("grupo").loc[grupo, "letra"], ha="center", fontsize=9)
eje.set(xlabel="Tratamiento", ylabel="Inhibición del crecimiento (%)",
        title="Actividad antifúngica · inhibición media ± DE")
eje.set_xticklabels(eje.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(CARPETA_SALIDA / "fig1_inhibicion.png", dpi=300, bbox_inches="tight")
fig.savefig(CARPETA_SALIDA / "fig1_inhibicion.pdf", bbox_inches="tight")
plt.show()

### Consigna 6.2 · Curvas dosis-respuesta

Graficá las curvas 4PL ajustadas en escala logarítmica de dosis, con los puntos
observados y la EC50 anotada. Guardá el resultado en `outputs/fig2_dosis_respuesta.png`.

**Se espera:** una figura con cuatro paneles (T1–T4) y la EC50 impresa en cada uno.


In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True)
malla = np.logspace(np.log10(20), np.log10(80), 100)

for eje, fila in zip(ejes.ravel(), ajustes_4pl.itertuples()):
    sub = grupos[grupos["familia"] == fila.familia].dropna(subset=["crecimiento_cm"])
    eje.scatter(sub["dosis_ug_ml"], sub["crecimiento_cm"], color="black", s=28, zorder=3)
    if fila.estimable:
        y_curva = curva_4pl(malla, fila.bottom_cm, CRECIMIENTO_CONTROL_CM, fila.ec50_ug_ml, fila.hill)
        eje.plot(malla, y_curva, color="#E45756", lw=2)
        eje.axvline(fila.ec50_ug_ml, color="0.5", ls="--", lw=1)
        eje.text(0.05, 0.85, f"EC50 ≈ {fila.ec50_ug_ml:g} µg/ml", transform=eje.transAxes, fontsize=9)
    else:
        eje.text(0.05, 0.85, "EC50 no estimable", transform=eje.transAxes, fontsize=9)
    eje.set_xscale("log")
    eje.set_title(fila.familia)
    eje.set_ylim(-0.05, 2.1)

for eje in ejes[1, :]:
    eje.set_xlabel("Dosis (µg/ml, escala log)")
for eje in ejes[:, 0]:
    eje.set_ylabel("Crecimiento (cm)")
fig.suptitle("Curvas dosis-respuesta (4PL, techo fijo en el control)")
fig.tight_layout()
fig.savefig(CARPETA_SALIDA / "fig2_dosis_respuesta.png", dpi=300, bbox_inches="tight")
plt.show()

### Consigna 6.3 · DLS en el tiempo

Graficá el tamaño y el índice de polidispersidad en `t0` y `t30` para T5–T8,
mostrando las barras de error de la desviación del tamaño.

**Se espera:** dos paneles, con `ND` visible como ausencia de dato en T7 y T8 a t30.


In [ ]:
fig, (eje_tam, eje_pdi) = plt.subplots(1, 2, figsize=(10, 4))
x = np.arange(len(dls_tidy))
ancho = 0.35

eje_tam.errorbar(x - ancho / 2, dls_tidy["tamano_t0_nm"], yerr=dls_tidy["desvio_t0_nm"],
                 fmt="o", capsize=3, label="t0")
eje_tam.errorbar(x + ancho / 2, dls_tidy["tamano_t30_nm"], yerr=dls_tidy["desvio_t30_nm"],
                 fmt="s", capsize=3, label="t30")
eje_tam.set(xticks=x, xticklabels=dls_tidy["tratamiento"],
            xlabel="Formulación", ylabel="Tamaño ± DE (nm)", title="Tamaño de partícula")
eje_tam.legend()

eje_pdi.scatter(x - ancho / 2, dls_tidy["pdi_t0"], label="t0", zorder=3)
eje_pdi.scatter(x + ancho / 2, dls_tidy["pdi_t30"], label="t30", zorder=3)
eje_pdi.set(xticks=x, xticklabels=dls_tidy["tratamiento"],
            xlabel="Formulación", ylabel="Índice de polidispersidad", title="Polidispersidad")
eje_pdi.legend()
fig.tight_layout()
fig.savefig(CARPETA_SALIDA / "fig3_dls.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Espectroscopía IR (bloque preparado)

Faltan los datos de IR de las formulaciones desde la 8 en adelante. Esta celda
busca archivos que coincidan con `PATRON_IR`; si no están, avisa y se salta sin
romper el cuaderno. Cuando lleguen, no hace falta cambiar el resto del análisis.


### Consigna 7.1 · Cargar si están disponibles

Buscá archivos con `Path('.').glob(PATRON_IR)`. Si hay, concatenalos en `datos_ir`;
si no, imprimí un aviso y dejá `datos_ir = None`.

**Se espera:** un mensaje claro sobre la ausencia de datos, sin excepciones.


In [ ]:
archivos_ir = sorted(Path(".").glob(PATRON_IR))

if not archivos_ir:
    datos_ir = None
    print("IR: todavía no hay archivos (formulaciones ≥ 8).")
    print(f"Patrón buscado: {PATRON_IR}. Esta celda se salta sin error; volvé a ejecutarla cuando lleguen.")
    for nombre_hoja, tabla in hojas_extra.items():
        print(f"Hoja adicional disponible: {nombre_hoja} ({tabla.shape[0]} filas)")
else:
    datos_ir = pd.concat([pd.read_csv(a) for a in archivos_ir], ignore_index=True)
    print("Archivos de IR encontrados:", [a.name for a in archivos_ir])
    display(datos_ir.head())

## 8. Verificación e informe

**Producto:** checks automáticos de integridad y un resumen escrito en
`outputs/informe_resumen.md`.


### Consigna 8.1 · Chequeos finales

Verificá que no haya negativos, que los faltantes declarados sigan siendo 3, que
los ceros sigan siendo 5, que DLS tenga 4 formulaciones y que las figuras existan.

**Se espera:** `Chequeos: OK` o un `AssertionError` que señale qué cambió.


In [ ]:
assert tidy["crecimiento_cm"].dropna().ge(0).all()
assert int(tidy["crecimiento_cm"].isna().sum()) == 3
assert int((tidy["crecimiento_cm"] == 0).sum()) == 5
assert len(dls_tidy) == 4
for nombre in ["fig1_inhibicion.png", "fig2_dosis_respuesta.png", "fig3_dls.png"]:
    assert (CARPETA_SALIDA / nombre).exists(), f"Falta {nombre}"
print("Chequeos: OK")

### Consigna 8.2 · Informe resumen

Escribí un resumen breve con el SHA-256, la estructura de datos, las decisiones de
tests y las advertencias de interpretación.

**Se espera:** `outputs/informe_resumen.md` creado y su contenido impreso.


In [ ]:
def df_a_markdown(df):
    """Tabla Markdown sin depender de tabulate."""
    columnas = [str(c) for c in df.columns]
    filas = [[str(v) for v in fila] for fila in df.itertuples(index=False, name=None)]
    ancho = [max(len(columnas[i]), *(len(f[i]) for f in filas)) if filas else len(columnas[i])
             for i in range(len(columnas))]
    separador = "| " + " | ".join("-" * a for a in ancho) + " |"
    cabecera = "| " + " | ".join(c.ljust(ancho[i]) for i, c in enumerate(columnas)) + " |"
    cuerpo = ["| " + " | ".join(v.ljust(ancho[i]) for i, v in enumerate(f)) + " |" for f in filas]
    return "\n".join([cabecera, separador] + cuerpo)

lineas = [
    "# Informe resumen — Actividad antifúngica",
    "",
    f"- Archivo: `{RUTA_EXCEL.name}` (SHA-256: `{sha256(RUTA_EXCEL)}`).",
    f"- Observaciones de actividad: {len(tidy)}; faltantes: {int(tidy['crecimiento_cm'].isna().sum())}; "
    f"ceros: {int((tidy['crecimiento_cm'] == 0).sum())}.",
    f"- Formulaciones en DLS: {len(dls_tidy)}.",
    f"- Semilla: {SEMILLA}; bootstrap: {N_BOOT} repeticiones.",
    "",
    "## Tests por familia",
    "",
    df_a_markdown(resultados),
    "",
    "## EC50 estimables",
    "",
    df_a_markdown(ajustes_4pl[ajustes_4pl["estimable"]][
        ["familia", "ec50_ug_ml", "ic_inf", "ic_sup", "r2"]])
    if ajustes_4pl["estimable"].any() else "Ninguna estimable.",
    "",
    "## Advertencias",
    "",
    "- El control tiene una sola medición (1.9 cm): todos los % de inhibición son condicionales a ese valor.",
    "- Hay n = 2 en T1 40, T5–T8 60 por celdas `NA`: la falta de significancia no implica ausencia de efecto.",
    "- T5–T8 tienen solo 2 dosis: EC50 no estimable por diseño.",
    "- Los ceros son inhibición completa, un dato válido (ver análisis de sensibilidad).",
    "- El análisis es exploratorio: los valores de p son descriptivos, no confirmatorios.",
    "- IR de formulaciones ≥ 8 pendiente; el bloque 7 ya está preparado.",
    "",
]

informe = "\n".join(lineas)
(CARPETA_SALIDA / "informe_resumen.md").write_text(informe, encoding="utf-8")
print(informe)